# Evaluation & Causal Reasoning

Building a model and trusting a model are different skills. A model with 97% accuracy on a held-out test set can be useless — or worse, actively harmful — if that accuracy is measured on the wrong distribution, with the wrong metric, or without an estimate of how much it might change on the next dataset. This notebook covers the reasoning framework that separates engineers who can fit models from engineers who can deploy them responsibly: (1) choosing metrics that reflect what actually matters, (2) understanding whether predicted probabilities correspond to real frequencies, (3) quantifying uncertainty in those metrics via bootstrap, (4) comparing models rigorously with A/B tests, and (5) reasoning about what would happen under a hypothetical intervention using causal inference.

## Classification Metrics: Precision, Recall, and ROC

Every binary classifier produces, for a given threshold $\tau \in [0, 1]$, a hard prediction $\hat{y} = \mathbf{1}[\hat{p}(x) \geq \tau].$ The four cells of the **confusion matrix** — true positives ($\text{TP}$), false positives ($\text{FP}$), false negatives ($\text{FN}$), true negatives ($\text{TN}$) — summarize how the classifier's predictions align with reality.

Two quantities derived from the confusion matrix are central to applied work. **Precision** is the fraction of positive predictions that are correct:

$$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}.$$

**Recall** (also called sensitivity or true positive rate) is the fraction of true positives that were detected:

$$\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}.$$

Precision and recall are in fundamental tension. Lowering the threshold $\tau$ recovers more positives (higher recall) at the cost of more false alarms (lower precision). The **F1 score** is their harmonic mean, which weights both equally:

$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}.$$

When the cost of a false negative differs from the cost of a false positive — as it usually does in high-stakes settings — the more general **$F_\beta$ score** places $\beta^2$ times as much weight on recall as on precision:

$$F_\beta = (1 + \beta^2) \cdot \frac{\text{Precision} \cdot \text{Recall}}{\beta^2 \cdot \text{Precision} + \text{Recall}}.$$

For $\beta > 1$, recall is favored (e.g. $\beta = 2$ for disease screening where missing a case is costly). For $\beta < 1$, precision is favored (e.g. spam filtering where false alarms degrade user experience).

**ROC curve.** The receiver operating characteristic (ROC) curve sweeps $\tau$ from $1$ down to $0$ and plots the true positive rate $\text{TPR} = \text{Recall}$ against the false positive rate $\text{FPR} = \text{FP}/(\text{FP} + \text{TN})$ at each threshold. The **area under the ROC curve** (AUC-ROC) admits a clean probabilistic interpretation: it is the probability that a randomly drawn positive example receives a higher predicted score than a randomly drawn negative example,

$$\text{AUC} = P\!\left(\hat{p}(x_+) > \hat{p}(x_-)\right),$$

where $x_+$ is drawn from the positive class and $x_-$ from the negative class. AUC-ROC equals $0.5$ for a random classifier and $1.0$ for a perfect ranker. Importantly, AUC-ROC is [threshold-free]{.underline}: it measures the model's ability to rank positives above negatives regardless of the operating point chosen.

**When to use PR curves.** The ROC curve can be misleadingly optimistic when classes are severely imbalanced, because the FPR denominator ($\text{FP} + \text{TN}$) is dominated by the large negative class, making FPR small even when $\text{FP}$ is large in absolute terms. The **precision-recall curve** (PR curve) avoids this: both axes depend on the positive class, so it gives an honest picture of retrieval quality. [When the positive class is rare, prefer the PR curve and its area (AUC-PR) over the ROC curve.]{.mark}

ROC and PR curves for logistic regression, random forest, and XGBoost on the breast cancer dataset:

In [ ]:
#| label: fig-roc-curves
#| fig-cap: "(**Left**) ROC curves for logistic regression, random forest, and XGBoost on the breast cancer dataset. AUC is shown in each legend entry; the dashed diagonal is the random-guessing baseline. (**Right**) Precision-recall curves for the same models. AUC-PR rewards retrieval quality and is more informative when positive examples are scarce."
#| code-fold: true

from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats("svg")

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from xgboost import XGBClassifier

# ── Data ──────────────────────────────────────────────────────────────────────
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

# ── Models ────────────────────────────────────────────────────────────────────
models = [
    ("Logistic Regression", LogisticRegression(max_iter=5000, random_state=0)),
    ("Random Forest",       RandomForestClassifier(n_estimators=200, random_state=0)),
    ("XGBoost",             XGBClassifier(n_estimators=200, use_label_encoder=False,
                                          eval_metric="logloss", random_state=0, verbosity=0)),
]

for name, clf in models:
    clf.fit(X_tr, y_tr)

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for i, (name, clf) in enumerate(models):
    scores = clf.predict_proba(X_te)[:, 1]

    # ROC
    fpr, tpr, _ = roc_curve(y_te, scores)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=f"C{i}", lw=2,
                 label=f"{name} (AUC = {roc_auc:.3f})")

    # PR
    prec, rec, _ = precision_recall_curve(y_te, scores)
    ap = average_precision_score(y_te, scores)
    axes[1].plot(rec, prec, color=f"C{i}", lw=2,
                 label=f"{name} (AP = {ap:.3f})")

# ROC diagonal baseline
axes[0].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.6, label="Random")
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve")
axes[0].legend(fontsize=8, loc="lower right")
axes[0].grid(linestyle="dotted", alpha=0.6)

axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend(fontsize=8, loc="lower left")
axes[1].grid(linestyle="dotted", alpha=0.6)

fig.tight_layout()
plt.show();

**Figure.** All three models achieve AUC-ROC above $0.99$ on this dataset, which is not particularly discriminating — the breast cancer dataset is moderate-difficulty with $569$ examples and $30$ features, and even logistic regression nearly saturates the ROC. The PR curve is similarly near-perfect here, confirming that the class balance ($\approx 63\%$ positive) does not pressure the metric.

## Calibration

A classifier is **calibrated** if its predicted probability $\hat{p}(x)$ matches the empirically observed frequency of the positive class among examples receiving that score. Formally, among all examples where the model predicts $\hat{p}(x) \approx p$, approximately a fraction $p$ should actually be positive. A perfectly calibrated model satisfies $P(Y = 1 \mid \hat{p}(X) = p) = p$ for all $p \in [0, 1].$

Calibration is visualized with a **reliability diagram**: predicted probabilities are binned into $M$ equal-width intervals, and the mean predicted confidence in each bin is plotted against the fraction of true positives in that bin. A perfectly calibrated model traces the diagonal.

**Expected Calibration Error.** The **ECE** aggregates the gap between confidence and accuracy across bins, weighted by the number of examples in each bin:

$$\text{ECE} = \sum_{m=1}^{M} \frac{|B_m|}{n} \left| \text{acc}(B_m) - \text{conf}(B_m) \right|,$$

where $|B_m|$ is the number of examples in bin $m$, $n$ is the total number of examples, $\text{acc}(B_m)$ is the fraction of true positives in bin $m$, and $\text{conf}(B_m)$ is the mean predicted probability in bin $m.$ A lower ECE indicates better calibration.

**Why certain models are poorly calibrated.** Logistic regression outputs probabilities that are, by construction, calibrated when the model is correctly specified — the cross-entropy loss directly penalizes miscalibration. Random forests, on the other hand, output leaf-node frequency estimates that cluster near $0$ and $1$ because trees are piecewise constant and fit training data closely. SVMs do not produce probabilities natively at all; their outputs are decision function scores, and any probability estimate requires a post-hoc sigmoid fit. Gradient boosted trees similarly tend to be overconfident because boosting pushes scores toward the extremes.

**Post-hoc calibration.** Two methods are standard. **Platt scaling** fits a logistic regression $\sigma(a \cdot f(x) + b)$ on the model's raw scores using a held-out calibration set; it is parametric and works well when the miscalibration is approximately sigmoidal. **Isotonic regression** fits a non-decreasing step function to map scores to calibrated probabilities; it is non-parametric, more flexible, but requires more data. Both are available in scikit-learn via [`CalibratedClassifierCV`](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibratedClassifierCV.html).

**When calibration matters.** For pure ranking tasks (information retrieval, ad ranking, recommendation), AUC-ROC and AUC-PR suffice — only the ordering of scores matters, not their absolute values. But when predicted probabilities feed decision-making — cost-benefit thresholds, expected value calculations, risk communication to clinicians — [calibration is essential]{.mark}. A model that outputs $\hat{p} = 0.8$ for cases that are actually positive only $30\%$ of the time will cause systematic errors in any downstream decision that treats $\hat{p}$ as a real probability.

Reliability diagrams before and after Platt scaling:

In [ ]:
#| label: fig-calibration-plot
#| fig-cap: "Reliability diagrams for logistic regression, random forest, and XGBoost before (solid) and after (dashed) Platt scaling. ECE is annotated for each model and condition. A perfectly calibrated model follows the diagonal."
#| code-fold: true

import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.model_selection import train_test_split

# ── Calibration set (hold out from training data) ─────────────────────────────
X_tr2, X_cal, y_tr2, y_cal = train_test_split(
    X_tr, y_tr, test_size=0.3, random_state=1, stratify=y_tr
)

def compute_ece(y_true, y_prob, n_bins=10):  # <1>
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            continue
        acc  = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / n) * abs(acc - conf)
    return ece

model_names = ["Logistic Regression", "Random Forest", "XGBoost"]
base_clfs   = [
    LogisticRegression(max_iter=5000, random_state=0),
    RandomForestClassifier(n_estimators=200, random_state=0),
    XGBClassifier(n_estimators=200, use_label_encoder=False,
                  eval_metric="logloss", random_state=0, verbosity=0),
]

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)

for ax, name, clf in zip(axes, model_names, base_clfs):
    # Fit on smaller training split, calibrate on calibration set
    clf.fit(X_tr2, y_tr2)
    cal_clf = CalibratedClassifierCV(clf, method="sigmoid", cv="prefit")  # <2>
    cal_clf.fit(X_cal, y_cal)

    # Scores on test set
    scores_raw = clf.predict_proba(X_te)[:, 1]
    scores_cal = cal_clf.predict_proba(X_te)[:, 1]

    # Calibration curves
    frac_pos_raw, mean_pred_raw = calibration_curve(y_te, scores_raw, n_bins=10)
    frac_pos_cal, mean_pred_cal = calibration_curve(y_te, scores_cal, n_bins=10)

    ece_raw = compute_ece(y_te, scores_raw)
    ece_cal = compute_ece(y_te, scores_cal)

    ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Perfect")
    ax.plot(mean_pred_raw, frac_pos_raw, "o-", color="C0", lw=2,
            label=f"Uncalibrated (ECE={ece_raw:.3f})")
    ax.plot(mean_pred_cal, frac_pos_cal, "s--", color="C1", lw=2,
            label=f"Platt scaled (ECE={ece_cal:.3f})")

    ax.set_title(name)
    ax.set_xlabel("Mean predicted probability")
    if ax is axes[0]:
        ax.set_ylabel("Fraction of positives")
    ax.legend(fontsize=7)
    ax.grid(linestyle="dotted", alpha=0.6)

fig.tight_layout()
plt.show();

Implementation notes:

1. `compute_ece` bins predicted probabilities into $M = 10$ equal-width bins and accumulates the weighted absolute gap between mean confidence and accuracy in each bin.
2. `cv="prefit"` instructs `CalibratedClassifierCV` to treat the classifier as already fitted and to learn only the calibration mapping on `X_cal`.

**Figure.** Logistic regression is already close to the diagonal, as expected from its well-specified probabilistic training objective. The random forest's uncalibrated curve bows toward the extremes — a characteristic S-shape caused by leaf frequencies clustering near $0$ and $1$. Platt scaling corrects this substantially. XGBoost shows a similar but often less severe distortion. After calibration, all three models sit much closer to the diagonal, with ECE reductions reflecting the improvement.

## Bootstrap Confidence Intervals

When we report a model's AUC on a test set of size $n$, we report a single number — a point estimate of the true AUC under the data-generating distribution. But point estimates without uncertainty quantification are dangerous in practice: an AUC of $0.873$ versus $0.869$ on the same test set may be pure noise. The **bootstrap** gives us a non-parametric way to estimate that uncertainty.

The procedure is simple: given a test set $\{(x_i, y_i)\}_{i=1}^n$, draw $B$ bootstrap resamples $\mathcal{D}^*_b$ of size $n$ with replacement. On each resample, compute the statistic of interest $\hat{\theta}^*_b$ (e.g., AUC). The empirical distribution $\{\hat{\theta}^*_1, \ldots, \hat{\theta}^*_B\}$ approximates the sampling distribution of $\hat{\theta}.$ The **percentile interval** at level $1 - \alpha$ is simply the $\alpha/2$ and $1 - \alpha/2$ quantiles of this empirical distribution:

$$\text{CI}_{1-\alpha} = \left[\hat{\theta}^*_{(\alpha/2)},\; \hat{\theta}^*_{(1-\alpha/2)}\right].$$

The **bias-corrected and accelerated (BCa) interval** is a refinement that corrects for (1) bias in the bootstrap distribution relative to the original estimate and (2) non-constant standard error of $\hat{\theta}$ (acceleration). BCa intervals have better theoretical coverage than plain percentile intervals, especially when the sampling distribution of $\hat{\theta}$ is skewed or when $n$ is small. In practice, $B = 1000$–$2000$ resamples suffices for stable interval endpoints.

**When to prefer bootstrap over asymptotic approximations.** Classical frequentist methods assume the estimator is asymptotically normal, which holds for many statistics under mild conditions. But AUC, partial AUC, F1, and differences of metrics are not obviously normal in finite samples, especially on imbalanced datasets with small positive counts. The bootstrap makes no distributional assumption and is straightforwardly applicable to any statistic. The cost is computational: $B$ evaluations of the metric.

**Application: CI on AUC and on a metric difference.** A common use case is asking whether model A is better than model B: compute $\hat{\delta} = \text{AUC}_A - \text{AUC}_B$ on each bootstrap resample, then check whether $0$ falls outside the confidence interval for $\delta.$ If the $95\%$ CI excludes $0$, the difference is significant at the $5\%$ level — no separate hypothesis test is needed.

Bootstrap confidence intervals on AUC for each model, and on the AUC difference between the best and second-best model:

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
B   = 2000  # bootstrap resamples

# Collect test-set scores for the three models
model_scores = {}
for name, clf in models:
    clf.fit(X_tr, y_tr)  # refit on full training split
    model_scores[name] = clf.predict_proba(X_te)[:, 1]

def bootstrap_auc_ci(y_true, y_score, B=2000, alpha=0.05, rng=None):  # <1>
    rng = rng or np.random.default_rng()
    n = len(y_true)
    boot_aucs = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, size=n)                                 # <2>
        boot_aucs[b] = roc_auc_score(y_true[idx], y_score[idx])
    lo = np.percentile(boot_aucs, 100 * alpha / 2)
    hi = np.percentile(boot_aucs, 100 * (1 - alpha / 2))
    return lo, hi, boot_aucs

print(f"{'Model':<25} {'AUC':>6}  {'95% CI':>20}")
print("-" * 55)
boot_distributions = {}
for name, scores in model_scores.items():
    lo, hi, dist = bootstrap_auc_ci(y_te, scores, B=B, rng=rng)
    auc_hat = roc_auc_score(y_te, scores)
    boot_distributions[name] = dist
    print(f"{name:<25} {auc_hat:.4f}  [{lo:.4f}, {hi:.4f}]")

1. `bootstrap_auc_ci` draws $B$ resamples with replacement and computes AUC on each, returning the percentile interval endpoints and the full bootstrap distribution.
2. `rng.integers(0, n, size=n)` samples row indices with replacement.

Bootstrap CI on the AUC difference between the top two models:

In [ ]:
import numpy as np

# AUC difference: XGBoost vs. Logistic Regression (as an example comparison)
names = list(model_scores.keys())
delta_dist = boot_distributions[names[2]] - boot_distributions[names[0]]  # XGB - LR

lo_d = np.percentile(delta_dist, 2.5)
hi_d = np.percentile(delta_dist, 97.5)
delta_hat = roc_auc_score(y_te, model_scores[names[2]]) - \
            roc_auc_score(y_te, model_scores[names[0]])

print(f"ΔAUC (XGBoost − Logistic Regression): {delta_hat:+.4f}")
print(f"95% bootstrap CI: [{lo_d:+.4f}, {hi_d:+.4f}]")
print(f"Zero excluded: {not (lo_d <= 0 <= hi_d)}")

## A/B Testing

Offline metrics — AUC, F1, RMSE — are proxies. The ground truth question is whether deploying model B instead of model A produces a better outcome for actual users, and the controlled experiment (A/B test) is the gold standard for answering it.

**Two-sample tests.** Given treatment (B) and control (A) groups of size $n_B$ and $n_A$ with sample means $\bar{X}_B, \bar{X}_A$ and sample variances $s_B^2, s_A^2$, the two-sample $t$-statistic is

$$t = \frac{\bar{X}_B - \bar{X}_A}{\sqrt{\dfrac{s_B^2}{n_B} + \dfrac{s_A^2}{n_A}}}.$$

Under the null hypothesis $H_0\colon \mu_B = \mu_A$, this follows approximately a $t$ distribution (or standard normal for large $n$). We reject $H_0$ when $|t| > t_{\alpha/2}$, where $\alpha$ is the significance level (typically $0.05$).

**Power and sample size.** Before running an experiment, we must decide how large it needs to be. The **power** $1 - \beta$ of a test is the probability of correctly rejecting $H_0$ when the true effect size is $\delta = \mu_B - \mu_A.$ For a two-sided $z$-test at significance $\alpha$ and power $1 - \beta$, the required sample size per group is

$$n \approx \frac{(z_{\alpha/2} + z_\beta)^2 \cdot 2\sigma^2}{\delta^2},$$

where $\sigma^2$ is the (assumed equal) population variance and $z_p = \Phi^{-1}(p).$ Plugging in $\alpha = 0.05$ and $1 - \beta = 0.80$ gives $z_{\alpha/2} + z_\beta \approx 1.96 + 0.84 = 2.80.$ The formula makes clear that detecting a small effect ($\delta$ small) requires a much larger sample, and that the required sample size grows quadratically as the minimum detectable effect shrinks.

**Multiple testing.** When we test $m$ hypotheses simultaneously (multiple metrics, multiple segments, multiple models), the probability of at least one false positive under the global null inflates rapidly. The **Bonferroni correction** controls the family-wise error rate (FWER) by using threshold $\alpha / m$ per test — conservative but simple. The **Benjamini-Hochberg procedure** controls the **false discovery rate** (FDR): sort the $m$ $p$-values $p_{(1)} \leq \ldots \leq p_{(m)}$ and reject all $H_{(k)}$ with $k \leq k^*$, where $k^* = \max\{k \colon p_{(k)} \leq \alpha k / m\}.$ FDR control is less conservative than FWER control, making it the right choice when a small fraction of false discoveries is acceptable.

**The peeking pitfall.** A common mistake in practice is to run the experiment, check results periodically, and stop early when $p < 0.05$. Under repeated peeking, the effective false positive rate is far above the nominal $\alpha$ — a $5\%$ test repeated $20$ times has nearly $100\%$ chance of a false positive somewhere.

:::{.callout-caution}
**Do not peek at running experiments.** Deciding to stop an A/B test early because $p < 0.05$ at an interim look invalidates the test's false positive guarantee. The actual $\alpha$ is much higher than the nominal level. Determine the sample size before the experiment and commit to a single analysis at the end.

:::

**Sequential testing.** When early stopping is operationally required — for example, to halt a harmful treatment quickly — **sequential testing** methods provide "always-valid" $p$-values: the type I error guarantee holds regardless of when you stop. The **mixture sequential probability ratio test** (mSPRT) and **e-values** are modern solutions. They replace the fixed-horizon $p$-value with a martingale process that can be monitored continuously without inflating $\alpha.$

Power analysis: required sample size per group as a function of minimum detectable effect size:

In [ ]:
#| code-fold: true

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

def required_n(delta, sigma=1.0, alpha=0.05, power=0.80):  # <1>
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta  = stats.norm.ppf(power)
    return int(np.ceil(2 * sigma**2 * (z_alpha + z_beta)**2 / delta**2))

deltas    = np.linspace(0.02, 0.5, 300)
ns_80     = [required_n(d, power=0.80) for d in deltas]
ns_90     = [required_n(d, power=0.90) for d in deltas]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(deltas, ns_80, color="C0", lw=2, label="Power = 0.80")
ax.plot(deltas, ns_90, color="C1", lw=2, label="Power = 0.90")
ax.axvline(0.1, color="gray", linestyle="dashed", lw=1, label="MDE = 0.10")
ax.set_xlabel("Minimum detectable effect $\\delta / \\sigma$")
ax.set_ylabel("Required $n$ per group")
ax.set_ylim(0, 5000)
ax.legend(fontsize=9)
ax.grid(linestyle="dotted", alpha=0.6)
fig.tight_layout()
plt.show();

1. `required_n` implements the standard two-sided $z$-test sample size formula with $\sigma$ as the assumed per-group standard deviation.

A concrete two-sample test and Benjamini-Hochberg correction across multiple metrics:

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(0)
N_A, N_B = 5000, 5000

# Simulate three metrics; treatment has a real effect only on metric 1
metrics = ["Conversion rate", "Session duration", "Bounce rate"]
true_effects = [0.02, 0.00, 0.00]  # only metric 0 has a real effect

pvals = []
for delta in true_effects:
    ctrl = rng.normal(loc=0.0,       scale=1.0, size=N_A)
    trt  = rng.normal(loc=delta,     scale=1.0, size=N_B)
    t, p = stats.ttest_ind(trt, ctrl)
    pvals.append(p)

# Benjamini-Hochberg FDR correction
def bh_correction(pvals, alpha=0.05):  # <1>
    m = len(pvals)
    order = np.argsort(pvals)
    thresholds = alpha * (np.arange(1, m + 1) / m)
    rejected = np.zeros(m, dtype=bool)
    for k in range(m - 1, -1, -1):  # find largest k where p_(k) <= alpha*k/m
        if pvals[order[k]] <= thresholds[k]:
            rejected[order[:k + 1]] = True
            break
    return rejected

rejected = bh_correction(pvals, alpha=0.05)

print(f"{'Metric':<20} {'p-value':>10}  {'BH reject':>12}")
print("-" * 46)
for metric, p, rej in zip(metrics, pvals, rejected):
    print(f"{metric:<20} {p:>10.4f}  {'✓' if rej else '✗':>12}")

1. The BH procedure sorts $p$-values, finds the largest rank $k$ where $p_{(k)} \leq \alpha k / m$, and rejects all hypotheses with smaller $p$-values. This controls the expected fraction of false discoveries among all rejections at level $\alpha.$

## Causal Inference: Propensity Scores and ATE

Correlation is not causation, and the distinction matters whenever we ask "what would happen if we intervened?" A model trained to predict churn might find that customers who call support are more likely to churn — not because support causes churn, but because unhappy customers both call support and churn. Acting on this correlation (routing customers away from support) would backfire. Causal reasoning forces us to be precise about the question.

**Potential outcomes framework.** The **Rubin causal model** formalizes causation using potential outcomes. For each unit $i$, define $Y_i(1)$ as the outcome that would be observed if unit $i$ received treatment ($T_i = 1$), and $Y_i(0)$ as the outcome under control ($T_i = 0$). The individual treatment effect is $Y_i(1) - Y_i(0).$ The **fundamental problem of causal inference** is that we observe only one of these for each unit — the factual outcome — and never the counterfactual.

The **average treatment effect** (ATE) averages over the population:

$$\tau = \mathbb{E}[Y(1) - Y(0)].$$

The **average treatment effect on the treated** (ATT) conditions on those who received treatment:

$$\tau_\text{ATT} = \mathbb{E}[Y(1) - Y(0) \mid T = 1].$$

In a randomized controlled trial, treatment assignment is independent of potential outcomes, so we can estimate $\tau$ by a simple difference in observed means. In **observational studies**, assignment is not random — units that receive treatment may differ systematically from controls. This **selection bias** means the naive difference $\mathbb{E}[Y \mid T=1] - \mathbb{E}[Y \mid T=0]$ conflates the treatment effect with differences due to **confounders**: pre-treatment covariates $X$ that influence both treatment assignment and the outcome.

**Propensity score.** Under the assumption of **strong ignorability** — $Y(0), Y(1) \perp\!\!\!\perp T \mid X$ (no unmeasured confounders) — it suffices to adjust for $X.$ The **propensity score** $e(X) = P(T = 1 \mid X)$ summarizes the confounders in a single number: if we condition on $e(X)$, treatment is as good as randomly assigned. In practice, we estimate $\hat{e}(X)$ by fitting logistic regression (or any classifier) on observed assignments.

**Propensity score matching** pairs each treated unit with one or more control units that have a similar $\hat{e}(X)$, then estimates $\tau$ from matched pairs. **Inverse probability weighting** (IPW) re-weights units to create a pseudo-population in which treatment is balanced:

$$\hat{\tau}_{\text{IPW}} = \frac{1}{n}\sum_{i=1}^n \frac{T_i Y_i}{\hat{e}(X_i)} - \frac{(1 - T_i)Y_i}{1 - \hat{e}(X_i)}.$$

IPW gives high weight to treated units with low propensity and to control units with high propensity — the "surprising" assignments that carry the most counterfactual information. If the propensity model is well-specified, $\hat{\tau}_{\text{IPW}}$ is a consistent estimator of the ATE.

**When strong ignorability fails.** If there are unmeasured confounders — variables that affect both treatment and outcome but are not recorded — neither matching nor IPW recovers the causal effect. One escape is an **instrumental variable** (IV): a variable $Z$ that affects treatment but affects the outcome only through treatment (the exclusion restriction). IV estimation recovers a local average treatment effect (LATE) for compliers — units whose treatment status is changed by the instrument.

**Setup.** We construct a synthetic dataset with a known ATE so we can evaluate how well IPW recovers it. Treatment assignment depends on a covariate $X_0$ (the confounder) so that naive comparison is biased:

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
n = 2000
TRUE_ATE = 2.0

# Covariates
X = rng.standard_normal((n, 3))

# Treatment assignment: probability depends on X[:,0] (the confounder)  # <1>
logit_e = 0.8 * X[:, 0]
e_true  = 1 / (1 + np.exp(-logit_e))
T       = rng.binomial(1, e_true).astype(float)

# Potential outcomes: Y(1) - Y(0) = TRUE_ATE for every unit               # <2>
noise   = rng.standard_normal(n)
Y0      = 3.0 * X[:, 0] + noise        # control outcome; confounded
Y1      = Y0 + TRUE_ATE
Y       = T * Y1 + (1 - T) * Y0        # observed outcome

# Naive estimator: biased because X[:,0] affects both T and Y
naive_ate = Y[T == 1].mean() - Y[T == 0].mean()
print(f"True ATE:          {TRUE_ATE:.3f}")
print(f"Naive ATE:         {naive_ate:.3f}  (biased by confounder)")

1. Treated units have higher values of $X_0$, and $X_0$ also raises the control outcome $Y(0)$. This creates positive confounding: the naive estimator overestimates the ATE.
2. We impose a constant treatment effect of $2.0$ so the ATE is exactly $2.0$ regardless of $X$.

Fitting the propensity score model and estimating ATE via IPW:

In [ ]:
from sklearn.linear_model import LogisticRegression

# Estimate propensity scores
ps_model = LogisticRegression(max_iter=1000)
ps_model.fit(X, T)
e_hat = ps_model.predict_proba(X)[:, 1]  # ê(X)

# Clip propensity scores to avoid extreme weights                         # <1>
e_hat_clipped = np.clip(e_hat, 0.05, 0.95)

# IPW estimator
ipw_weights = T / e_hat_clipped - (1 - T) / (1 - e_hat_clipped)         # <2>
ipw_ate = ipw_weights @ Y / n

print(f"True ATE:          {TRUE_ATE:.3f}")
print(f"Naive ATE:         {naive_ate:.3f}")
print(f"IPW ATE:           {ipw_ate:.3f}")

1. Propensity scores near $0$ or $1$ produce extreme weights and high variance. Clipping to $[0.05, 0.95]$ trades a small amount of bias for substantially lower variance — a practical necessity in observational studies with limited overlap.
2. The IPW formula assigns each treated observation weight $1/\hat{e}(x_i)$ and each control observation weight $-1/(1 - \hat{e}(x_i))$, then averages the weighted outcomes.

**Result.** The naive estimator is biased — it overestimates the ATE because treated units have higher baseline outcomes $Y(0)$ due to confounding by $X_0.$ The IPW estimator corrects for this by re-weighting observations according to the inverse of their propensity score, recovering the true ATE substantially more accurately.

:::{.callout-note}
IPW assumes the propensity model is correctly specified and that strong ignorability holds. If important confounders are unobserved or the propensity model is misspecified, IPW will still be biased. The **doubly robust** estimator — which combines IPW with an outcome model — is consistent if either the propensity model or the outcome model is correctly specified, providing an extra layer of protection against misspecification.

:::

Bootstrap confidence interval on the IPW ATE estimate:

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

rng_boot = np.random.default_rng(7)
B = 1000
boot_ates = np.empty(B)

for b in range(B):
    idx = rng_boot.integers(0, n, size=n)
    Xb, Tb, Yb = X[idx], T[idx], Y[idx]

    ps = LogisticRegression(max_iter=1000).fit(Xb, Tb)
    eh = np.clip(ps.predict_proba(Xb)[:, 1], 0.05, 0.95)
    boot_ates[b] = (Tb / eh - (1 - Tb) / (1 - eh)) @ Yb / len(Yb)

lo_ate, hi_ate = np.percentile(boot_ates, [2.5, 97.5])
print(f"IPW ATE:   {ipw_ate:.3f}")
print(f"95% CI:    [{lo_ate:.3f}, {hi_ate:.3f}]")
print(f"True ATE ({TRUE_ATE:.1f}) within CI: {lo_ate <= TRUE_ATE <= hi_ate}")

---

■